# Week 6: Explainability, Collinearity Consolidation & Grounded GenAI Narratives

**Project:** Fraud Risk Analytics & Detection System  
**Supporting Capability:** Grounded GenAI Analyst Explanations  
**Architecture:** Champion LightGBM $\rightarrow$ TreeSHAP $\rightarrow$ $V$-Collinearity Consolidation $\rightarrow$ Business Action Policy $\rightarrow$ Grounded Narrative Generator $\rightarrow$ Automated Grounding Validator  

---

### Core Objectives:
1. **TreeSHAP Feature Attribution**: Quantify exact feature contributions in log-odds margin space on held-out temporal evaluation transactions ($N=1,500$).
2. **Collinearity Consolidation**: Consolidate 162 near-duplicate collinear $V$-feature pairs ($|r| \ge 0.98$ identified in Week 1) into human-interpretable risk driver clusters.
3. **Business Action Policy Mapping**: Map model probabilities into operational risk tiers (`LOW`, `MEDIUM`, `HIGH`) and predefined actions (`APPROVE`, `STEP_UP_AUTH`, `MANUAL_REVIEW`).
4. **Grounded Narrative Generation**: Generate structured analyst summaries offline using local open-source LLMs (Ollama / deterministic template / optional Grok API).
5. **Automated Grounding Validator**: Audit all generated text against direct and derived evidence with automatic fallback-on-rejection safeguards.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.explainability.shap_explainer import FraudSHAPExplainer
from src.explainability.reason_codes import ReasonCodeEngine, BusinessDecisionPolicy
from src.explainability.narrative_generator import GroundedNarrativeGenerator
from src.explainability.llm_client import LLMClient, DeterministicTemplateProvider
from src.validation.grounding_validator import GroundingValidator

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
print("Libraries and project explainability modules successfully loaded.")

## 1. Load Champion Model & Held-Out Demo Slice

We load the Champion LightGBM classifier (`models/champion_model.joblib`) and the curated held-out test transaction slice (`data/processed/demo_replay_slice.parquet`).

In [ ]:
explainer = FraudSHAPExplainer()
demo_df = pd.read_parquet(project_root / "data" / "processed" / "demo_replay_slice.parquet")

print(f"Held-out demo slice shape: {demo_df.shape}")
print(f"Champion Base Value (log-odds): {explainer.base_value:.4f}")
print(f"Total features in model: {len(explainer.feature_names)}")

## 2. Global SHAP Feature Importances

We compute the mean absolute SHAP values across held-out transactions to identify the top global drivers of model decisions.

In [ ]:
importance_df = explainer.compute_global_feature_importance(demo_df, max_samples=1500)
top_20 = importance_df.head(20)

plt.figure(figsize=(12, 8))
palette = sns.color_palette("Blues_r", n_colors=20)
sns.barplot(data=top_20, x="mean_abs_shap", y="feature", palette=palette)
plt.title("Global TreeSHAP Feature Attribution (Top 20 Features on Held-Out Test Slice)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Mean Absolute SHAP Value (Log-Odds Impact)", fontsize=12)
plt.ylabel("Model Feature", fontsize=12)
plt.tight_layout()
plt.show()

## 3. $V$-Feature Collinearity Consolidation in Action

In Week 1, we identified 162 near-duplicate collinear $V$-feature pairs ($|r| \ge 0.98$). Below we demonstrate how our `ReasonCodeEngine` clusters redundant $V$-features (e.g. $V95, V101, V279, V293$) into single interpretable concepts rather than overwhelming analysts with 4 duplicate lines.

In [ ]:
engine = ReasonCodeEngine()
test_feat_names = ["V95", "V101", "V279", "amt_zscore_card1", "TransactionAmt", "C1"]
test_feat_values = [8.0, 8.0, 8.0, 3.8, 450.0, 15.0]
test_shap_values = np.array([1.45, 1.42, 1.38, 1.10, 0.65, -0.50])

top_risk, top_mitigating = engine.consolidate_and_extract_reason_codes(
    feature_names=test_feat_names,
    feature_values=test_feat_values,
    shap_values=test_shap_values,
    top_k=5,
)

print("=== CONSOLIDATED REASON CODES ===")
print("\nPrimary Risk Drivers:")
for rc in top_risk:
    print(f" - [{rc.category}] {rc.display_name}: Observed={rc.feature_value}, SHAP=+{rc.shap_value:.3f} ({rc.contribution_pct:.1f}% impact)")
    if rc.is_collinear_cluster:
        print(f"   * Consolidated Cluster Members: {rc.cluster_members[:5]}... (Duplicate V-features suppressed)")

print("\nMitigating Factors:")
for rc in top_mitigating:
    print(f" - [{rc.category}] {rc.display_name}: Observed={rc.feature_value}, SHAP={rc.shap_value:.3f}")

## 4. Case Study 1: High-Risk Transaction with Grounded Narrative & Validator Audit

Let us inspect a genuine high-risk transaction ($p \ge 0.70$), compute its TreeSHAP attribution, generate the structured analyst narrative, and audit the output with the `GroundingValidator`.

In [ ]:
# Find a high-risk transaction in demo slice
high_risk_candidates = demo_df[demo_df["isFraud"] == 1]
sample_tx = high_risk_candidates.iloc[0]

payload = explainer.explain_transaction(sample_tx, top_k=5)
generator = GroundedNarrativeGenerator(preferred_provider="deterministic")
gen_result = generator.generate_narrative_for_payload(payload)

print(gen_result.narrative_text)
print("\n" + "=" * 60)
print(" GROUNDING VALIDATION AUDIT:")
print(f" Is Grounded:              {gen_result.grounding_validation.is_grounded}")
print(f" Grounding Score:          {gen_result.grounding_validation.grounding_score:.3f}")
print(f" Direct Facts Checked:     {gen_result.grounding_validation.direct_facts_checked}")
print(f" Derived Facts Verified:   {gen_result.grounding_validation.derived_facts_verified}")
print(f" Fallback Substituted:     {gen_result.is_fallback_substituted}")
print("=" * 60)

## 5. Case Study 2: Borderline Medium-Risk Transaction (`STEP_UP_AUTH`)

Transactions in the range $0.10 \le p < 0.35$ trigger step-up authentication (3DS, OTP, or identity verification) rather than blocking or manual review.

In [ ]:
# Filter for a medium risk transaction
probs = explainer.model.predict_proba(explainer._prepare_features(demo_df))[:, 1]
med_idx = np.where((probs >= 0.15) & (probs <= 0.30))[0][0]
sample_med_tx = demo_df.iloc[med_idx]

med_payload = explainer.explain_transaction(sample_med_tx, top_k=5)
med_result = generator.generate_narrative_for_payload(med_payload)

print(med_result.narrative_text)
print("\n" + "=" * 60)
print(f" Risk Tier:         {med_payload.predicted_risk_tier}")
print(f" Decision Action:   {med_payload.decision_action}")
print(f" Grounding Valid:   {med_result.grounding_validation.is_grounded}")
print("=" * 60)

## 6. Grounding Validator Safeguard Demonstration

We demonstrate how the `GroundingValidator` actively catches hallucinations, fabricated numbers, speculative assertions (e.g. "dark web syndicates"), and directional inversions.

In [ ]:
validator = GroundingValidator()

# 1. Test clean grounded narrative
clean_text = DeterministicTemplateProvider.format_from_payload(payload.to_dict())
clean_audit = validator.validate_narrative(clean_text, payload.to_dict())
print(f"1. Clean Narrative Audit:         is_grounded={clean_audit.is_grounded}, score={clean_audit.grounding_score:.2f}")

# 2. Test hallucinated numbers and speculative phrases
corrupted_text = clean_text + "\n- The card was confirmed stolen on the dark web for $99,999.00 in a criminal syndicate attack."
corrupted_audit = validator.validate_narrative(corrupted_text, payload.to_dict())
print(f"2. Corrupted Narrative Audit:     is_grounded={corrupted_audit.is_grounded}, score={corrupted_audit.grounding_score:.2f}")
print(f"   * Rejection Reasons:           {corrupted_audit.rejection_reasons}")
print(f"   * Speculation Violations:      {corrupted_audit.speculation_violations}")
print(f"   * Unsupported Numbers:         {corrupted_audit.unsupported_numbers}")

## 7. Week 6 Batch Validation Summary Report

We load and visualize the final validation report generated across all 1,500 held-out demo transactions.

In [ ]:
with open(project_root / "data" / "processed" / "grounding_validation_report.json", "r") as f:
    report = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Risk tier distribution pie
tiers = report["risk_tier_distribution"]
ax1.pie(tiers.values(), labels=tiers.keys(), autopct="%1.1f%%", colors=["#2ecc71", "#f39c12", "#e74c3c"], startangle=140, explode=(0.02, 0.02, 0.05))
ax1.set_title("Risk Tier Distribution (N=1,500 Demo Slice)", fontsize=13, fontweight="bold")

# Action distribution bar
actions = report["decision_action_distribution"]
sns.barplot(x=list(actions.keys()), y=list(actions.values()), ax=ax2, palette="viridis")
ax2.set_title("Decision Actions Triggered by Model Policy", fontsize=13, fontweight="bold")
ax2.set_ylabel("Transaction Volume")

plt.tight_layout()
plt.show()

print("=== SUMMARY VALIDATION AUDIT REPORT ===")
print(json.dumps(report, indent=2))